In [1]:
#Cellule 1 : Connexion Azure ML 
from azure.ai.ml import MLClient, Input, Output, command 
from azure.ai.ml.dsl import pipeline 
from azure.ai.ml.constants import AssetTypes 
from azure.ai.ml.entities import Environment 
from azure.identity import DefaultAzureCredential 

# Connexion automatique depuis config.json 
ml_client = MLClient.from_config(DefaultAzureCredential()) 

print(f'Workspace : {ml_client.workspace_name}') 
print(f'Resource Group : {ml_client.resource_group_name}') 
print(f'Subscription : {ml_client.subscription_id}') 

# Vérification du compute 
for c in ml_client.compute.list(): 
    print(f'Compute disponible : {c.name} ({c.type})')

Found the config file in: /config.json


Workspace : ws-pauline2
Resource Group : p.thomieres-rg
Subscription : 7eb0fbeb-3ad3-4759-b3b8-3682f7313123
Compute disponible : instance-pauline (computeinstance)
Compute disponible : cluster-pauline (amlcompute)


In [2]:
# Cellule 2 : Créer l'Environment depuis conda.yml 
from azure.ai.ml.entities import Environment 

env = Environment( 
    name='olist-train-env', 
    version='4', 
    description='Environnement pour le pipeline Olist', 
    conda_file='./conda.yml', 
    image='mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest', ) 
    
# Enregistrer dans le workspace 
env = ml_client.environments.create_or_update(env)
print(f'Environment créé : {env.name} v{env.version}')
print('Note : le build de l\'image Docker peut prendre 5 à 10 min (première fois)')

Environment créé : olist-train-env v4
Note : le build de l'image Docker peut prendre 5 à 10 min (première fois)


In [2]:
# Cellule 3 : Vérification des Data Assets 
assets_needed = ['olist-orders', 'olist-order-items', 'olist-products'] 

for name in assets_needed:
    try:
        asset = ml_client.data.get(name, version='1') 
        print(f'✓ {asset.name:30s} v{asset.version} → {asset.path[:50]}...')
    except Exception as e: 
        print(f'✗ MANQUANT : {name} — retourner à l\'étape 6')

✓ olist-orders                   v1 → azureml://subscriptions/7eb0fbeb-3ad3-4759-b3b8-36...
✓ olist-order-items              v1 → azureml://subscriptions/7eb0fbeb-3ad3-4759-b3b8-36...
✓ olist-products                 v1 → azureml://subscriptions/7eb0fbeb-3ad3-4759-b3b8-36...


In [3]:
# Cellule 4 : Import des composants — VERSION ADAPTEE
import sys, os

 
project_root = os.path.abspath('')
if project_root not in sys.path:
    sys.path.insert(0, project_root)
 
# Test que CommandComponent est disponible
from azure.ai.ml.entities import CommandComponent
from azure.ai.ml import Input, Output
from azure.ai.ml.constants import AssetTypes
 
# Importer les composants depuis src/
# Ils sont maintenant des objets CommandComponent, pas des fonctions
try:
    from src.preprocess import preprocess_olist
    print('✓ preprocess_olist importe')
except ImportError as e: 
    print(f'✗ preprocess.py : {e}')

try:
    from src.feature_engineering import feature_engineering_olist
    print('✓ feature_engineering importe')
except ImportError as e: 
    print(f'✗ preprocess.py : {e}')

try:
    from src.train import train_olist
    print('✓ train importé')
except ImportError as e: 
    print(f'✗ train.py : {e}')

try:
    from src.evaluate import evaluate_olist
    print('✓ evaluate importé')
except ImportError as e:
    print(f'✗ evaluate.py : {e}')

✓ preprocess_olist importe
✓ feature_engineering importe
✓ train importé
✓ evaluate importé


In [4]:
# Cellule 5 : Charger la définition du pipeline
try:
    from pipeline import olist_pipeline
    print('✓ pipeline olist_pipeline chargé')
    print('Prêt à instancier avec vos hyperparamètres.')
except ImportError as e:
    print(f'✗ pipeline.py : {e}')

✓ pipeline olist_pipeline chargé
Prêt à instancier avec vos hyperparamètres.


In [ ]:
# Cellule 6 : Instancier le pipeline et soumettre
import datetime
 
# ── Récupérer les Data Assets ────────────────────────────────────────────
def get_asset(name, version='1'):
    asset = ml_client.data.get(name, version=version)
    return Input(type=AssetTypes.URI_FILE, path=asset.path)
 
orders   = get_asset('olist-orders')
items    = get_asset('olist-order-items')
products = get_asset('olist-products')
 
# ── Vos choix ici ────────────────────────────────────────────────────────
# Modifier ces valeurs pour explorer différentes configurations
MODEL_TYPE   = 'ridge'   # random_forest | gradient_boosting | lightgbm | ridge
N_ESTIMATORS = 500
MAX_DEPTH    = 10
LEARN_RATE   = 0.1
 
# ── Instancier le pipeline ────────────────────────────────────────────────
pipeline_job = olist_pipeline(
    raw_orders=orders,
    raw_items=items,
    raw_products=products,
    model_type=MODEL_TYPE,
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    learning_rate=LEARN_RATE,
)
 
# Nommer le run pour le retrouver dans Studio
ts = datetime.datetime.now().strftime('%m%d-%H%M')
pipeline_job.experiment_name = 'olist-delivery-days'
pipeline_job.display_name    = f'{MODEL_TYPE}-n{N_ESTIMATORS}-d{MAX_DEPTH}-{ts}'
 
# ── Soumettre ────────────────────────────────────────────────────────────
submitted_job = ml_client.jobs.create_or_update(pipeline_job)
 
print(f'Job soumis       : {submitted_job.name}')
print(f'Nom affiché      : {submitted_job.display_name}')
print(f'Expérience       : {submitted_job.experiment_name}')
print(f'')
print(f'URL Studio       : {submitted_job.studio_url}')
print(f'')
print('Ouvrir l\'URL dans le navigateur pour suivre le graphe.')

Job soumis       : goofy_roti_jydxv7qq6r
Nom affiché      : gradient_boosting-n500-d10-0330-1010
Expérience       : olist-delivery-days

URL Studio       : https://ml.azure.com/runs/goofy_roti_jydxv7qq6r?wsid=/subscriptions/7eb0fbeb-3ad3-4759-b3b8-3682f7313123/resourcegroups/p.thomieres-rg/workspaces/ws-pauline2&tid=f017b958-d19a-4cb1-b4e7-03a430980b51

Ouvrir l'URL dans le navigateur pour suivre le graphe.


In [8]:
# Cellule 7 : Suivre les logs en temps réel (optionnel) 
# # Décommenter pour attendre la fin du pipeline 

# ml_client.jobs.stream(submitted.name) 
# Ou vérifier le statut sans bloquer : 
import time 
job = ml_client.jobs.get(submitted_job.name) 
print(f'Statut actuel : {job.status}') 
print(f'Pour voir les détails : {job.studio_url}') 

# Récupérer les métriques quand c'est terminé : 
# (Exécuter cette cellule après Completed) 
# ml_client.jobs.download( 
#   name=submitted.name, 
#   download_path='./pipeline_outputs/',
#   output_name='metrics' 
# )

Statut actuel : Running
Pour voir les détails : https://ml.azure.com/runs/khaki_cheetah_pvtrynysw8?wsid=/subscriptions/7eb0fbeb-3ad3-4759-b3b8-3682f7313123/resourcegroups/p.thomieres-rg/workspaces/ws-pauline2&tid=f017b958-d19a-4cb1-b4e7-03a430980b51
